17)
Mechanically, self-attention computes queries, keys, and values from the same sequence, whereas cross-attention computes queries from one sequence and keys/values from a different sequence. In this project, self-attention uses text tokens as queries, keys, and values, while cross-attention uses text tokens as queries and image patch embeddings as keys and values.

Semantically, self-attention allows tokens within a sequence to exchange information and understand their relationships, whereas cross-attention enables one modality to retrieve relevant information from another modality. In the multimodal captioning model, cross-attention allows the text decoder to access visual information from image patches while generating the caption.



18)
The causal self-attention layer is masked because the decoder is generating text autoregressively. When predicting the next character or token, the model should only have access to previously generated text and not future tokens. The causal mask prevents information leakage from future positions.

The cross-attention layer is not causally masked because the entire image is available from the beginning. Unlike text generation, there is no concept of a "future image patch." The decoder should be able to attend to any image patch when generating any token in the caption.

The MLP layer does not use attention and therefore does not require masking. It simply applies learned nonlinear transformations independently to each token representation.

If the self-attention layer were not causal, the model could see future text tokens during training and would effectively cheat, leading to unrealistic generation at inference time. If cross-attention were made causal, the decoder would only be able to access a subset of image patches, unnecessarily restricting access to visual information and reducing captioning performance.

19)
In cross-attention, the output sequence length equals the query length because an output vector is produced for every query position. If the query tensor has shape (B, Tq, d) and the context tensor has shape (B, Tc, d), the attention weights have shape (B, Tq, Tc). Multiplying these weights by the value vectors produces an output of shape (B, Tq, d). The context length Tc is used to compute attention scores, but it is summed over during the weighted aggregation, so the final output retains the query length Tq rather than the context length.

20)
If the vision encoder and text decoder use different embedding dimensions, their features cannot be directly used together in cross-attention because the query, key, and value projections must operate in compatible feature spaces. For example, a text embedding of dimension 128 cannot be directly compared with an image embedding of dimension 192.

One solution is to use the same embedding dimension for both the vision encoder and text decoder (e.g., n_embd = 128). A second solution is to add a learned projection layer, such as a linear transformation, that maps the vision features into the decoder embedding space (or vice versa) before applying cross-attention.

21)
The cross-attention heatmap showed that the model attended more strongly to a small number of image patches, particularly patches near the center of the image. This suggests that the decoder relied on specific visual regions when generating captions. Since CIFAR-10 objects are usually centered and the task only involves predicting simple class names, the model did not need highly detailed or localized attention patterns to perform reasonably well.

22)
A real multimodal model that is similar to Task 4 is LLaVA. Both models combine a vision encoder with a language model and use visual features to help generate text. In my implementation, a Vision Transformer (ViT) encodes the image into patch embeddings, and a text decoder uses cross-attention to access those image features while generating a caption.

The main difference is scale and capability. My model was trained on CIFAR-10 with simple synthetic captions such as "this is a cat," whereas LLaVA is trained on much larger datasets containing images and natural language instructions. As a result, LLaVA can answer complex questions about images, while my model can only generate simple class-based captions.

